# प्राथमिकताका सदस्य मध्यवर्ती सफ्टवेयरसहित होटल बुकिंग

यो नोटबुकले Microsoft Agent Framework प्रयोग गरी **फंक्शन-आधारित मध्यवर्ती सफ्टवेयर** देखाउँछ। हामी सर्तमूलक कार्यप्रवाहको उदाहरणलाई विस्तार गरी एउटा मध्यवर्ती तह थप्छौं जसले प्राथमिकताका सदस्यहरूलाई विशेष अधिकार दिन्छ।

## तपाईं के सिक्नुभयो:
1. **फंक्शन-आधारित मध्यवर्ती सफ्टवेयर**: फंक्शन परिणामहरू अवरोध गर्ने र परिमार्जन गर्ने
2. **सन्दर्भ पहुँच**: कार्यान्वयन पछि `context.result` पढ्ने र परिमार्जन गर्ने
3. **व्यावसायिक तर्क कार्यान्वयन**: प्राथमिकताका सदस्य लाभहरू
4. **परिणाम ओभरराइड**: प्रयोगकर्ता अवस्थाको आधारमा फंक्शन नतिजा परिवर्तन गर्ने
5. **एकै कार्यप्रवाह, फरक नतिजा**: मध्यवर्ती सफ्टवेयर-चालित व्यवहार परिवर्तनहरू

## मध्यवर्ती सफ्टवेयर सहितको कार्यप्रवाह वास्तुकला:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## सर्तमूलक कार्यप्रवाहबाट मुख्य भिन्नता:

**मध्यवर्ती सफ्टवेयर बिना** (14-conditional-workflow.ipynb):
- पेरिसमा कोठा छैन → वैकल्पिक_agent तर्फ मार्गनिर्देशन

**मध्यवर्ती सफ्टवेयर सहित** (यो नोटबुक):
- नियमित प्रयोगकर्ता + पेरिस → कोठा छैन → वैकल्पिक_agent तर्फ मार्गनिर्देशन
- प्राथमिकता प्रयोगकर्ता + पेरिस → 🌟 मध्यवर्ती सफ्टवेयरले ओभरराइड गर्छ! → उपलब्ध → बुकिंग_agent तर्फ मार्गनिर्देशन

## आवश्यकताहरू:
- Microsoft Agent Framework स्थापना गरिएको
- सर्तमूलक कार्यप्रवाहको बुझाइ (हेर्नुहोस् 14-conditional-workflow.ipynb)
- GitHub टोकन वा OpenAI API कुञ्जी
- मध्यवर्ती सफ्टवेयर ढाँचाहरुको आधारभूत बुझाइ


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## चरण १: संरचित आउटपुटहरूको लागि Pydantic मोडेलहरू परिभाषित गर्नुहोस्

यी मोडेलहरूले **स्किमा** परिभाषित गर्छन् जुन एजेन्टहरूले फर्काउनेछन्। हामीले मिडलवेयरले उपलब्धता नतिजा परिवर्तन गर्दा ट्र्याक गर्न `priority_override` फील्ड थपेका छौं।


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## चरण २: प्राथमिकता सदस्यहरूको डाटाबेस परिभाषित गर्नुहोस्

यस डेमोका लागि, हामी प्राथमिकता सदस्यता डाटाबेसको नक्कल गर्नेछौं। उत्पादनमा, यसले वास्तविक डाटाबेस वा API लाई सोध्नेछ।

**प्राथमिकता सदस्यहरू:**
- `alice@example.com` - भिआईपी सदस्य
- `bob@example.com` - प्रिमियम	member  
- `priority_user` - परीक्षण खाता


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## चरण 3: होटेल बुकिंग टूल बनाउनुहोस्

सशर्त कार्यप्रवाह जस्तै, तर अब यसलाई मिडलवेयरले अवरुद्ध गर्नेछ!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## चरण ४: 🌟 प्राथमिकता जाँच मिडलवेयर बनाउनुहोस् (मुख्य सुविधा!)

यो नोटबुकको **मुख्य कार्यक्षमता** हो। मिडलवेयरले:

१. **होटल_बुकिङ** कार्य कललाई रोक्छ
२. `next(context)` कल गरेर सामान्य रूपमा कार्यलाई चलाउँछ
३. `context.result` मा नतिजा जाँच गर्दछ
४. यदि प्रयोगकर्ता प्राथमिकता छ र कोठा उपलब्ध छैन भने नतिजा ओभरराइड गर्दछ
५. संशोधित नतिजा एजेन्टलाई फर्काउँछ

**मुख्य ढाँचा:**
```python
async def my_middleware(context, next):
    await next(context)  # फङ्क्सन चलाउनुहोस्
    # अहिले context.result मा फङ्क्सनको नतिजा छ
    if some_condition:
        context.result = new_value  # ओभरराइड गर्नुहोस्!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## चरण ५: राउटिंगका लागि सर्त कार्यहरू परिभाषित गर्नुहोस्

सर्त कार्यहरू त्यस्तै जस्तै सर्त workflows मा हुन्छन् - तिनीहरूले संरचित आउटपुट जाँचेर राउटिंग निर्धारण गर्छन्।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## चरण ६: कस्टम प्रदर्शन कार्यान्वयनकर्ता सिर्जना गर्नुहोस्

पहिले जस्तै कार्यान्वयनकर्ता - अन्तिम कार्यप्रवाह आउटपुट प्रदर्शन गर्छ।


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## चरण ७: वातावरण चरहरू लोड गर्नुहोस्

LLM क्लाइन्ट कन्फिगर गर्नुहोस् (Microsoft Foundry / Azure OpenAI)। 


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## चरण ८: मिडलवेयर सहित AI एजेन्टहरू सिर्जना गर्नुहोस्

**मुख्य भिन्नता:** availability_agent सिर्जना गर्दा, हामी `middleware` प्यारामिटर पास गर्छौं!

यसरी हामी एजेन्टको फंक्शन इन्भोकेसन पाईपलाइनमा priority_check_middleware इन्जेक्ट गर्छौं।


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## चरण ९: कार्यप्रवाह निर्माण गर्नुहोस्

पहिले जस्तो नै कार्यप्रवाह संरचना - उपलब्धतामा आधारित सशर्त रूटिङ।


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## चरण १०: परीक्षण केस १ - पेरिस मा नियमित प्रयोगकर्ता (ओभरराइड छैन)

एक नियमित प्रयोगकर्ताले पेरिस बुक गर्न प्रयास गर्छ → कोठा छैन → alternative_agent तर्फ मार्गनिर्देशन गर्दछ


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## चरण 11: परीक्षण केस 2 - 🌟 प्राथमिकता प्रयोगकर्ता पेरिसमा (ओभरराइडसहित!)

एक प्राथमिकता सदस्यले पेरिस बुकिङ गर्ने प्रयास गर्छ → सुरुमा कोठा छैन → 🌟 मिडलवेयरले ओभरराइड गर्यो! → booking_agent तर्फ मार्गनिर्देशन

**यो मिडलवेयर शक्तिको मुख्य प्रदर्शन हो!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## चरण १२: परीक्षण मामला ३ - स्टकहोममा प्राथमिकता प्रयोगकर्ता (पहिले देखि उपलब्ध)

प्राथमिकता प्रयोगकर्ता स्टकहोम प्रयास गर्छ → कोठा उपलब्ध छन् → कुनै ओभरराइड आवश्यक छैन → booking_agent मा मार्गनिर्देशन

यसले देखाउँछ कि मध्यस्थ सफ्टवेयर मात्र आवश्यक परेमा काम गर्दछ!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## मुख्य तत्त्वहरू र मिडलवेयर अवधारणाहरू

### ✅ तपाईंले के सिक्नुभयो:

#### **1. फंक्शन-आधारित मिडलवेयर ढाँचा**

मिडलवेयरले साधारण async फंक्शन प्रयोग गरेर फंक्शन कलहरूलाई रोक्छ:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # कार्यसम्पादन अघि
    print("Intercepting...")
    
    # कार्यसम्पादन गर्नुहोस्
    await next(context)
    
    # कार्यसम्पादन पछि - परिणाम निरीक्षण गर्नुहोस्
    if context.result:
        # आवश्यक भए परिणाम परिमार्जन गर्नुहोस्
        context.result = modified_value
```

#### **2. सन्दर्भ पहुँच र परिणाम ओभरराइड**

- `context.function` - कल भइरहेको फंक्शनमा पहुँच
- `context.arguments` - फंक्शनको तर्क पढ्नुहोस्
- `context.kwargs` - अतिरिक्त प्यारामिटरहरूमा पहुँच
- `await next(context)` - फंक्शन कार्यान्वयन गर्नुहोस्
- `context.result` - फंक्शनको आउटपुट पढ्नु/परिमार्जन गर्नुहोस्

#### **3. व्यापारिक तर्क कार्यान्वयन**

हाम्रो मिडलवेयर प्राथमिक सदस्य लाभहरू लागू गर्छ:
- **नियमित प्रयोगकर्ता**: कुनै परिवर्तन छैन, मानक कार्यप्रवाह
- **प्राथमिकता प्रयोगकर्ता**: "उपलब्धता छैन" → "उपलब्ध" ओभरराइड गर्नुहोस्
- **सशर्त तर्क**: आवश्यक पर्दा मात्र ओभरराइड गर्दछ

#### **4. उस्तै कार्यप्रवाह, फरक परिणामहरू**

मिडलवेयरको शक्ति:
- ✅ कार्यप्रवाह संरचनामा कुनै परिवर्तन छैन
- ✅ उपकरण फंक्शनमा कुनै परिवर्तन छैन
- ✅ सशर्त मार्गनिर्देशन तर्कमा कुनै परिवर्तन छैन
- ✅ केवल मिडलवेयर → फरक व्यवहार!

### 🚀 वास्तविक विश्व उपयोगहरू:

1. **VIP/प्रीमियम सुविधाहरू**
   - प्रिमियम प्रयोगकर्ताहरूका लागि दर सीमा ओभरराइड गर्नुहोस्
   - स्रोतहरूमा प्राथमिकता पहुँच प्रदान गर्नुहोस्
   - गतिशील रूपमा प्रिमियम सुविधाहरू अनलक गर्नुहोस्

2. **A/B परीक्षण**
   - प्रयोगकर्ताहरूलाई फरक कार्यान्वयनहरूमा रुट गर्नुहोस्
   - विशिष्ट प्रयोगकर्ताहरूसँग नयाँ सुविधाहरू परीक्षण गर्नुहोस्
   - क्रमिक रूपमा सुविधा रोलआउटहरू

3. **सुरक्षा र अनुपालन**
   - फंक्शन कलहरूको अडिट गर्नुहोस्
   - संवेदनशील अपरेसनहरू अवरुद्ध गर्नुहोस्
   - व्यापार नियमहरू लागू गर्नुहोस्

4. **प्रदर्शन अनुकूलन**
   - विशिष्ट प्रयोगकर्ताहरूका लागि परिणामहरू क्यास गर्नुहोस्
   - सम्भव भएमा महंगा अपरेसनहरू स्किप गर्नुहोस्
   - गतिशील स्रोत आवंटन

5. **त्रुटि ह्यान्डलिंग र पुन: प्रयास**
   - त्रुटिहरूलाई सजिलै समात्नुहोस् र सम्हाल्नुहोस्
   - पुन: प्रयास तर्क कार्यान्वयन गर्नुहोस्
   - वैकल्पिक कार्यान्वयनतर्फ फर्कनुहोस्

6. **लगिङ र अनुगमन**
   - फंक्शन कार्यान्वयन समयहरू ट्र्याक गर्नुहोस्
   - प्यारामिटरहरू र परिणामहरू लग गर्नुहोस्
   - प्रयोग ढाँचाहरू अनुगमन गर्नुहोस्

### 🔑 डेकोरेटरहरूबाट मुख्य भिन्नताहरू:

| सुविधा | डेकोरेटर | मिडलवेयर |
|---------|-----------|------------|
| **क्षेत्र** | एकल फंक्शन | एजेन्टका सबै फंक्शनहरू |
| **लचिलोपन** | परिभाषामा स्थिर | रनटाइममा गतिशील |
| **सन्दर्भ** | सीमित | पूर्ण एजेन्ट सन्दर्भ |
| **संयोजन** | धेरै डेकोरेटरहरू | मिडलवेयर पाइपलाइन |
| **एजेन्ट-सचेत** | होइन | हो (एजेन्ट अवस्थामा पहुँच) |

### 📚 मिडलवेयर कहिले प्रयोग गर्ने:

✅ **मिडलवेयर प्रयोग गर्नुहोस् जब:**
- तपाईंले प्रयोगकर्ता/सेसन स्थितिमा आधारित व्यवहार परिमार्जन गर्न आवश्यक छ
- तपाईंले धेरै फंक्शनमा तर्क लागू गर्न चाहानुहुन्छ
- तपाईंलाई एजेन्ट-स्तर सन्दर्भमा पहुँच चाहिन्छ
- तपाईं क्रस-कटिङ चासोहरू लागू गर्दै हुनुहुन्छ (लगिङ, प्रमाणीकरण, आदि)

❌ **मिडलवेयर प्रयोग नगर्नुहोस् जब:**
- साधारण इनपुट मान्यकरण (Pydantic प्रयोग गर्नुहोस्)
- फंक्शन-विशेष तर्क (फंक्शनमा राख्नुहोस्)
- एकपटकका परिमार्जनहरू (प्रत्यक्ष फंक्शन परिवर्तन गर्नुहोस्)

### 🎓 उन्नत ढाँचाहरू:

```python
# बहु मिडलवेयर (प्रवाहितिको क्रम महत्त्वपूर्ण छ!)
middleware=[
    logging_middleware,      # सुरुमा लगहरू
    auth_middleware,         # त्यसपछि प्रमाणीकरण जाँच गर्दछ
    cache_middleware,        # त्यसपछि क्याच जाँच गर्दछ
    rate_limit_middleware,   # त्यसपछि दर सीमा लगाउँछ
    priority_check_middleware  # अन्तमा प्राथमिकता जाँच
]

# सर्तसहित मिडलवेयर कार्यान्वयन
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # नतिजा संशोधन गर्नुहोस्
    else:
        # कार्यान्वयन पूर्ण रूपमा छोड्नुहोस्
        context.result = cached_value
```

### 🔗 सम्बन्धित अवधारणाहरू:

- **एजेन्ट मिडलवेयर**: agent.run() कलहरू रोक्छ
- **फंक्शन मिडलवेयर**: उपकरण फंक्शन कलहरू रोक्छ (हामीले प्रयोग गरेका!)
- **मिडलवेयर पाइपलाइन**: मिडलवेयरको श्रृंखला जसले क्रमबद्ध रूपमा चल्दछ
- **सन्दर्भ प्रवाह**: मिडलवेयर श्रृंखलाबाट अवस्था पास गर्ने


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
